# End-to-End ML Inference Pipeline
This notebook demonstrates how the machine learning pipeline processes an uploaded file (CSV). 

It covers:
1. **Ingestion**: Loading an uploaded dataset.
2. **Validation**: Checking if the file contains the required sensor features.
3. **Preprocessing**: Scaling the features using our saved `StandardScaler`.
4. **Prediction**: Generating anomaly predictions using our saved Machine Learning model.
5. **Output**: Returning the final dataset with failure predictions appended.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import warnings
warnings.filterwarnings('ignore')

## 1. Define the Inference Pipeline Class
We encapsulate our prediction logic into a reusable class. This is identical to the logic used behind the scenes when a user uploads a CSV file through the Streamlit web dashboard.

In [ ]:
class MLPredictionPipeline:
    def __init__(self, model_path='models/best_model.pkl', scaler_path='models/scaler.pkl'):
        """Initialize the pipeline by loading the pre-trained artifacts."""
        print(f"Loading model from: {model_path}")
        self.model = joblib.load(model_path)
        
        print(f"Loading scaler from: {scaler_path}")
        self.scaler = joblib.load(scaler_path)
        
        # The exact order of features the model was trained on
        self.required_features = ['temperature', 'vibration', 'pressure', 'current', 'voltage', 'rpm', 'humidity', 'load']

    def process_and_predict(self, file_path):
        """
        Reads an uploaded CSV file, validates it, scales features, and predicts.
        """
        print(f"\n--- Starting Pipeline for: {file_path} ---")
        
        # 1. Ingestion
        df = pd.read_csv(file_path)
        print(f"Successfully loaded dataset with {len(df)} rows.")
        
        # 2. Validation
        missing_cols = [col for col in self.required_features if col not in df.columns]
        if missing_cols:
            raise ValueError(f"Uploaded file is missing required columns: {missing_cols}")
            
        # Extract only the required features in the correct order
        X_input = df[self.required_features]
        
        # 3. Preprocessing (Standardization)
        print("Applying StandardScaler preprocessing...")
        X_scaled = self.scaler.transform(X_input)
        
        # 4. Prediction
        print("Generating predictions...")
        predictions = self.model.predict(X_scaled)
        
        if hasattr(self.model, 'predict_proba'):
            probabilities = self.model.predict_proba(X_scaled)[:, 1]
        else:
            probabilities = [None] * len(predictions)
            
        # 5. Output Preparation
        result_df = df.copy()
        result_df['Predicted_Failure'] = predictions
        result_df['Failure_Probability'] = probabilities
        
        # Helper column for interpretation
        result_df['Maintenance_Action'] = result_df['Predicted_Failure'].apply(
            lambda x: 'Schedule Maintenance ⚠️' if x == 1 else 'No Action Required ✅'
        )
        
        print("--- Pipeline Execution Complete ---")
        return result_df

## 2. Simulate User Upload
Let's create a temporary CSV file acting as the "Uploaded file" by the user.

In [ ]:
# Create a mock uploaded file representing new unseen data
mock_upload = pd.DataFrame([
    {'machine_id': 'M-701', 'temperature': 68.0, 'vibration': 3.1, 'pressure': 105.0, 'current': 16.0, 'voltage': 220.0, 'rpm': 1480, 'humidity': 46.0, 'load': 82.0}, # Normal
    {'machine_id': 'M-702', 'temperature': 88.5, 'vibration': 7.4, 'pressure': 135.0, 'current': 24.0, 'voltage': 212.0, 'rpm': 2900, 'humidity': 55.0, 'load': 96.0}, # Anomalous
    {'machine_id': 'M-703', 'temperature': 70.2, 'vibration': 4.0, 'pressure': 110.0, 'current': 17.5, 'voltage': 218.0, 'rpm': 1550, 'humidity': 48.0, 'load': 85.0}  # Normal
])

upload_path = 'temp_uploaded_data.csv'
mock_upload.to_csv(upload_path, index=False)
print(f"Simulated file uploaded to: {upload_path}")

## 3. Run the File through the Pipeline
We initialize our pipeline system and push the uploaded file path through it.

In [ ]:
# Initialize the pipeline (Loads models only once)
pipeline = MLPredictionPipeline()

# Process the uploaded file
final_predictions_df = pipeline.process_and_predict(upload_path)

# Display the results to the User / Dashboard
display(final_predictions_df[['machine_id', 'Maintenance_Action', 'Failure_Probability', 'temperature', 'vibration']])

## 4. Cleaning up (Optional)
We can save the predicted dataset to a new CSV file for the user to download, and delete the temporary uploaded file.

In [ ]:
# Save the resulting dataframe to a downloadable CSV
output_path = 'prediction_results.csv'
final_predictions_df.to_csv(output_path, index=False)
print(f"Prediction results saved for download at: {output_path}")

# Clean up the temporary upload
if os.path.exists(upload_path):
    os.remove(upload_path)
    print("Temporary uploaded file removed.")